In [10]:
def upgma(n, D):
    # 初始化 clusters：每个叶子是一个簇
    clusters = {i: [i] for i in range(n)}

    # 初始化年龄 Age
    Age = {i: 0.0 for i in range(n)}

    # 邻接表（最终输出）
    adj = {}

    # 当前可用的内部节点编号
    next_node = n

    # 主循环：直到只剩一个簇
    while len(clusters) > 1:

        # 1. 找到距离最小的两个簇
        min_dist = float("inf")
        pair = None
        keys = list(clusters.keys())

        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                a, b = keys[i], keys[j]
                if D[a][b] < min_dist:
                    min_dist = D[a][b]
                    pair = (a, b)

        Ci, Cj = pair

        # 2. 创建新节点
        Cnew = next_node
        next_node += 1

        # 合并簇
        clusters[Cnew] = clusters[Ci] + clusters[Cj]

        # 设置新节点年龄
        Age[Cnew] = D[Ci][Cj] / 2.0

        # 在邻接表中添加结构
        adj.setdefault(Cnew, [])
        adj.setdefault(Ci, [])
        adj.setdefault(Cj, [])

        adj[Cnew].append((Ci, None))
        adj[Ci].append((Cnew, None))
        
        adj[Cnew].append((Cj, None))
        adj[Cj].append((Cnew, None))


        # 3. 更新距离矩阵
        D[Cnew] = {}
        for k in D:
            if k not in (Ci, Cj, Cnew):
                size_i = len(clusters[Ci])
                size_j = len(clusters[Cj])
                size_new = size_i + size_j

                D[Cnew][k] = (size_i * D[Ci][k] + size_j * D[Cj][k]) / size_new
                D[k][Cnew] = D[Cnew][k]

        # 删除旧簇的行列
        for k in list(D.keys()):
            if k in (Ci, Cj):
                del D[k]
            else:
                if Ci in D[k]:
                    del D[k][Ci]
                if Cj in D[k]:
                    del D[k][Cj]

        # 删除旧簇
        del clusters[Ci]
        del clusters[Cj]

    # 4. 计算边长
    for parent in adj:
        new_list = []
        for child, _ in adj[parent]:
            weight = Age[parent] - Age[child]
            new_list.append((child, weight))
        adj[parent] = new_list


    return adj


# ----------- 输入输出处理（符合题目要求） -----------

# 输入
n = 4
matrix = [
    [0, 20, 17, 11],
    [20, 0, 20, 13],
    [17, 20, 0, 10],
    [11, 13, 10, 0]
]

# 转换成字典格式
D = {i: {} for i in range(n)}
for i in range(n):
    for j in range(n):
        D[i][j] = float(matrix[i][j])

# 运行 UPGMA
adj = upgma(n, D)

# 输出
for u in sorted(adj.keys()):
    for v, w in adj[u]:
        print(f"{u}->{v}:{abs(w):.3f}")


0->5:7.000
1->6:8.833
2->4:5.000
3->4:5.000
4->2:5.000
4->3:5.000
4->5:2.000
5->0:7.000
5->4:2.000
5->6:1.833
6->1:8.833
6->5:1.833


In [12]:
n = 27
matrix_str = """0 733 782 1031 1248 1275 1122 775 1068 947 1390 829 934 1065 1102 1048 1290 755 1306 1382 1351 1227 1051 1232 849 1393 1224 
733 0 957 1077 1297 743 909 1186 1340 1083 1011 929 1109 1352 922 1110 1338 1094 897 838 1402 1362 1125 1339 1210 1141 898 
782 957 0 753 1184 1387 1243 1037 834 1027 1181 1257 730 987 854 1153 985 828 1038 740 1113 1364 870 1373 1326 1276 877 
1031 1077 753 0 1396 867 783 853 1183 825 1358 1446 1313 1009 969 1197 880 1425 1282 1135 1206 1334 1415 1445 1067 1379 1118 
1248 1297 1184 1396 0 1179 1262 890 1371 919 780 1112 1039 1399 1457 872 1100 1283 1005 1097 959 1418 1193 814 1119 928 951 
1275 743 1387 867 1179 0 738 1328 1421 1450 1386 1384 1246 1089 1277 1247 796 1093 1392 1191 821 1452 1041 1033 806 1196 896 
1122 909 1243 783 1262 738 0 1172 902 731 1318 850 1327 746 979 900 1159 1347 776 1320 971 1335 1331 1208 1180 1017 1426 
775 1186 1037 853 890 1328 1172 0 1055 842 868 1239 1438 1115 862 884 1439 1162 1023 1053 774 1374 1222 1148 1004 1345 1157 
1068 1340 834 1183 1371 1421 902 1055 0 803 757 1178 754 1175 1075 1215 942 1057 903 936 1293 1287 1028 982 989 1453 1070 
947 1083 1027 825 919 1450 731 842 803 0 1292 1369 1319 1124 863 788 1278 835 837 1044 1203 1456 781 1025 1442 1267 1052 
1390 1011 1181 1358 780 1386 1318 868 757 1292 0 1143 1407 1086 1303 1021 1169 920 1145 905 917 1081 756 1238 1388 964 1295 
829 929 1257 1446 1112 1384 850 1239 1178 1369 1143 0 1344 1366 1131 878 1349 1177 1249 859 1368 830 949 1198 921 1008 767 
934 1109 730 1313 1039 1246 1327 1438 754 1319 1407 1344 0 1378 1449 1423 916 895 1174 946 1301 1260 1288 1182 1185 965 955 
1065 1352 987 1009 1399 1089 746 1115 1175 1124 1086 1366 1378 0 1385 1096 1133 759 1138 1343 1063 1294 1409 1263 1325 1188 1398 
1102 922 854 969 1457 1277 979 862 1075 863 1303 1131 1449 1385 0 930 939 1255 1270 1058 1151 1061 729 1410 1353 1348 785 
1048 1110 1153 1197 872 1247 900 884 1215 788 1021 878 1423 1096 930 0 1132 1234 832 1337 980 1207 1035 1105 841 1346 925 
1290 1338 985 880 1100 796 1159 1439 942 1278 1169 1349 916 1133 939 1132 0 1090 1321 823 954 836 1354 1316 1040 1271 861 
755 1094 828 1425 1283 1093 1347 1162 1057 835 920 1177 895 759 1255 1234 1090 0 1252 1420 1279 1315 799 875 1098 1285 976 
1306 897 1038 1282 1005 1392 776 1023 903 837 1145 1249 1174 1138 1270 832 1321 1252 0 791 871 1200 1433 848 1091 1372 1356 
1382 838 740 1135 1097 1191 1320 1053 936 1044 905 859 946 1343 1058 1337 823 1420 791 0 1187 1417 1363 953 745 983 1006 
1351 1402 1113 1206 959 821 971 774 1293 1203 917 1368 1301 1063 1151 980 954 1279 871 1187 0 1144 904 840 933 918 1146 
1227 1362 1364 1334 1418 1452 1335 1374 1287 1456 1081 830 1260 1294 1061 1207 836 1315 1200 1417 1144 0 1176 773 879 851 1250 
1051 1125 870 1415 1193 1041 1331 1222 1028 781 756 949 1288 1409 729 1035 1354 799 1433 1363 904 1176 0 1152 1024 792 858 
1232 1339 1373 1445 814 1033 1208 1148 982 1025 1238 1198 1182 1263 1410 1105 1316 875 848 953 840 773 1152 0 1259 1209 744 
849 1210 1326 1067 1119 806 1180 1004 989 1442 1388 921 1185 1325 1353 841 1040 1098 1091 745 933 879 1024 1259 0 1012 950 
1393 1141 1276 1379 928 1196 1017 1345 1453 1267 964 1008 965 1188 1348 1346 1271 1285 1372 983 918 851 792 1209 1012 0 1166 
1224 898 877 1118 951 896 1426 1157 1070 1052 1295 767 955 1398 785 925 861 976 1356 1006 1146 1250 858 744 950 1166 0"""

matrix = [list(map(int, line.split())) for line in matrix_str.splitlines()]

# 转换成字典格式
D = {i: {} for i in range(n)}
for i in range(n):
    for j in range(n):
        D[i][j] = float(matrix[i][j])

# 运行 UPGMA
adj = upgma(n, D)

# 输出
for u in sorted(adj.keys()):
    for v, w in adj[u]:
        print(f"{u}->{v}:{abs(w):.3f}")

0->30:366.500
1->30:366.500
2->28:365.000
3->37:402.000
4->40:441.250
5->36:398.000
6->29:365.500
7->35:387.000
8->33:378.500
9->29:365.500
10->33:378.500
11->38:415.000
12->28:365.000
13->34:379.500
14->27:364.500
15->39:416.000
16->36:398.000
17->34:379.500
18->39:416.000
19->32:372.500
20->35:387.000
21->38:415.000
22->27:364.500
23->31:372.000
24->32:372.500
25->41:464.750
26->31:372.000
27->14:364.500
27->22:364.500
27->46:155.750
28->2:365.000
28->12:365.000
28->43:107.750
29->6:365.500
29->9:365.500
29->37:36.500
30->0:366.500
30->1:366.500
30->43:106.250
31->23:372.000
31->26:372.000
31->40:69.250
32->19:372.500
32->24:372.500
32->44:110.000
33->8:378.500
33->10:378.500
33->46:141.750
34->13:379.500
34->17:379.500
34->47:142.625
35->7:387.000
35->20:387.000
35->42:82.750
36->5:398.000
36->16:398.000
36->44:84.500
37->3:402.000
37->29:36.500
37->45:99.125
38->11:415.000
38->21:415.000
38->41:49.750
39->15:416.000
39->18:416.000
39->42:53.750
40->4:441.250
40->31:69.250
40->48:90

In [20]:
def neighbor_joining(D, nodes, next_node):
    n = len(nodes)

    # Base case: only two nodes left
    if n == 2:
        i, j = nodes
        return {
            i: [(j, D[i][j])],
            j: [(i, D[i][j])]
        }

    # Compute total distances
    total = {i: sum(D[i][j] for j in nodes) for i in nodes}

    # Build Q-matrix
    Q = {}
    for i in nodes:
        Q[i] = {}
        for j in nodes:
            if i == j:
                Q[i][j] = 0
            else:
                Q[i][j] = (n - 2) * D[i][j] - total[i] - total[j]

    # Find pair (i, j) minimizing Q
    min_val = float("inf")
    pair = None
    for i in nodes:
        for j in nodes:
            if i != j and Q[i][j] < min_val:
                min_val = Q[i][j]
                pair = (i, j)

    i, j = pair

    # Compute limb lengths
    delta = (total[i] - total[j]) / (n - 2)
    limb_i = (D[i][j] + delta) / 2
    limb_j = (D[i][j] - delta) / 2

    # Create new node m
    m = next_node
    next_node += 1

    # Build new distance matrix
    new_nodes = [x for x in nodes if x not in (i, j)] + [m]
    D_new = {x: {} for x in new_nodes}

    for x in new_nodes:
        for y in new_nodes:
            if x == m and y == m:
                D_new[x][y] = 0
            elif x == m:
                D_new[x][y] = (D[i][y] + D[j][y] - D[i][j]) / 2
            elif y == m:
                D_new[x][y] = (D[i][x] + D[j][x] - D[i][j]) / 2
            else:
                D_new[x][y] = D[x][y]

    # Recursive call
    T = neighbor_joining(D_new, new_nodes, next_node)

    # Add edges for i and j
    T.setdefault(m, [])
    T.setdefault(i, [])
    T.setdefault(j, [])

    T[m].append((i, limb_i))
    T[i].append((m, limb_i))

    T[m].append((j, limb_j))
    T[j].append((m, limb_j))

    return T


def run_nj(matrix):
    n = len(matrix)

    # Convert matrix to dict
    D = {i: {} for i in range(n)}
    for i in range(n):
        for j in range(n):
            D[i][j] = float(matrix[i][j])

    nodes = list(range(n))

    # Run NJ
    T = neighbor_joining(D, nodes, next_node=n)

    # Output adjacency list
    for u in sorted(T.keys()):
        # sort neighbors by node index
        T[u] = sorted(T[u], key=lambda x: x[0])
        for v, w in T[u]:
            print(f"{u}->{v}:{w:.3f}")


In [22]:
matrix_str = """0 1448 1687 1883 1368 2029 1865 1979 1736 1838 2005 1898 1500 1528 1034 1809 1101 1791 1625 1995 1703 1575 1981 1352 1364 1933 1358 1747 1685 2038 1652 1131 
1448 0 1393 1894 1859 1851 1195 1777 1057 1044 1713 1548 1960 1129 1248 1688 1277 1039 1690 1641 1046 1176 1052 1223 1645 2045 1359 1402 1772 1198 1964 2017 
1687 1393 0 1182 1476 1522 1255 1292 1356 1155 1604 1523 1030 1051 1370 1436 1483 2033 1086 1178 1696 1640 1081 1507 1571 1893 1914 1029 1816 1487 1419 1820 
1883 1894 1182 0 1582 1477 1574 1579 1288 1244 1531 1629 1619 1721 1909 1795 1567 1080 1488 1513 1558 1070 1390 1102 1216 1760 1908 1375 1076 1056 1589 1183 
1368 1859 1476 1582 0 1646 1492 1576 1704 1776 1998 2001 1676 1972 1961 1627 1591 1162 1643 2037 1435 1427 1705 1451 1912 1107 1740 2010 1362 1873 1852 1465 
2029 1851 1522 1477 1646 0 1274 1050 1609 2026 1269 1187 1311 1094 1068 1471 1948 1626 2039 1962 1160 1662 1226 1457 1631 1808 1568 1281 1431 1782 1800 1706 
1865 1195 1255 1574 1492 1274 0 1902 1032 1756 1284 1860 1054 2020 1586 1212 1263 1896 1399 1193 1622 1875 1525 1337 2036 1514 1470 1605 1968 1485 1421 1830 
1979 1777 1292 1579 1576 1050 1902 0 1230 1884 1167 2043 1839 1737 1911 1426 1078 1110 1372 1870 1391 1241 1185 1489 1299 1055 1100 1654 1585 1374 1763 1770 
1736 1057 1356 1288 1704 1609 1032 1230 0 1958 1049 1827 1222 1540 1757 1691 1913 1572 2011 1614 1897 1542 1355 1157 1214 1338 1335 1491 1510 1888 1821 1853 
1838 1044 1155 1244 1776 2026 1756 1884 1958 0 1822 1564 1577 1725 1497 1929 1553 1243 1075 1367 1636 1389 2015 1446 1235 1322 1456 1048 1903 1750 1754 1385 
2005 1713 1604 1531 1998 1269 1284 1167 1049 1822 0 1270 1346 1341 2012 1403 1414 1712 1467 1639 1469 1745 1387 1874 1256 1134 1250 1432 1973 1208 1767 1824 
1898 1548 1523 1629 2001 1187 1860 2043 1827 1564 1270 0 1726 1236 1127 1474 1405 1204 1868 1590 1351 1648 1799 1695 1570 1861 1126 1971 1850 1651 1928 1689 
1500 1960 1030 1619 1676 1311 1054 1839 1222 1577 1346 1726 0 2027 1290 1334 1308 1247 1395 1555 1140 1970 1258 1880 1759 1857 1801 1454 1642 1119 1819 1378 
1528 1129 1051 1721 1972 1094 2020 1737 1540 1725 1341 1236 2027 0 1940 1188 1264 1042 1317 1137 1172 1065 1242 1293 1563 1228 1363 1608 1461 1825 1412 1180 
1034 1248 1370 1909 1961 1068 1586 1911 1757 1497 2012 1127 1290 1940 0 2018 1409 1287 1444 2048 1278 1751 1560 1361 1420 1701 2028 1084 1494 1305 1939 1105 
1809 1688 1436 1795 1627 1471 1212 1426 1691 1929 1403 1474 1334 1188 2018 0 1283 1024 1593 2035 1885 1975 1280 2009 1667 1630 1125 1602 1584 1381 1844 1562 
1101 1277 1483 1567 1591 1948 1263 1078 1913 1553 1414 1405 1308 1264 1409 1283 0 1452 1541 1802 1953 1866 2047 1533 1422 1060 1462 1315 1366 1074 1114 1987 
1791 1039 2033 1080 1162 1626 1896 1110 1572 1243 1712 1204 1247 1042 1287 1024 1452 0 1752 1769 1312 1856 1146 1588 1408 1407 1779 1396 1206 1189 1826 1683 
1625 1690 1086 1488 1643 2039 1399 1372 2011 1075 1467 1868 1395 1317 1444 1593 1541 1752 0 1136 1565 1624 1320 1733 1561 1692 1190 1266 1234 1539 2030 1177 
1995 1641 1178 1513 2037 1962 1193 1870 1614 1367 1639 1590 1555 1137 2048 2035 1802 1769 1136 0 1138 1727 1272 1194 1753 1289 1481 1201 1439 1197 1343 1085 
1703 1046 1696 1558 1435 1160 1622 1391 1897 1636 1469 1351 1140 1172 1278 1885 1953 1312 1565 1138 0 1806 1041 1698 1945 1722 1124 1566 1716 1245 1499 2046 
1575 1176 1640 1070 1427 1662 1875 1241 1542 1389 1745 1648 1970 1065 1751 1975 1866 1856 1624 1727 1806 0 1394 1991 1674 1990 1400 1143 1997 1429 1037 1931 
1981 1052 1081 1390 1705 1226 1525 1185 1355 2015 1387 1799 1258 1242 1560 1280 2047 1146 1320 1272 1041 1394 0 1607 1301 1508 1179 1994 1530 1653 1524 2013 
1352 1223 1507 1102 1451 1457 1337 1489 1157 1446 1874 1695 1880 1293 1361 2009 1533 1588 1733 1194 1698 1991 1607 0 1210 1458 1159 1921 1309 1331 1464 2023 
1364 1645 1571 1216 1912 1631 2036 1299 1214 1235 1256 1570 1759 1563 1420 1667 1422 1408 1561 1753 1945 1674 1301 1210 0 1846 1678 1610 1918 1550 1694 1115 
1933 2045 1893 1760 1107 1808 1514 1055 1338 1322 1134 1861 1857 1228 1701 1630 1060 1407 1692 1289 1722 1990 1508 1458 1846 0 1581 1680 1273 1215 1321 1118 
1358 1359 1914 1908 1740 1568 1470 1100 1335 1456 1250 1126 1801 1363 2028 1125 1462 1779 1190 1481 1124 1400 1179 1159 1678 1581 0 1876 1069 1811 1205 1059 
1747 1402 1029 1375 2010 1281 1605 1654 1491 1048 1432 1971 1454 1608 1084 1602 1315 1396 1266 1201 1566 1143 1994 1921 1610 1680 1876 0 1788 1259 1490 1397 
1685 1772 1816 1076 1362 1431 1968 1585 1510 1903 1973 1850 1642 1461 1494 1584 1366 1206 1234 1439 1716 1997 1530 1309 1918 1273 1069 1788 0 1778 1993 1635 
2038 1198 1487 1056 1873 1782 1485 1374 1888 1750 1208 1651 1119 1825 1305 1381 1074 1189 1539 1197 1245 1429 1653 1331 1550 1215 1811 1259 1778 0 1209 1033 
1652 1964 1419 1589 1852 1800 1421 1763 1821 1754 1767 1928 1819 1412 1939 1844 1114 1826 2030 1343 1499 1037 1524 1464 1694 1321 1205 1490 1993 1209 0 1369 
1131 2017 1820 1183 1465 1706 1830 1770 1853 1385 1824 1689 1378 1180 1105 1562 1987 1683 1177 1085 2046 1931 2013 2023 1115 1118 1059 1397 1635 1033 1369 0"""

matrix = [list(map(float, line.split())) for line in matrix_str.splitlines()]

run_nj(matrix)

0->32:578.600
1->45:517.974
2->41:468.077
3->38:483.776
4->34:630.438
5->39:559.723
6->37:523.430
7->39:490.277
8->37:508.570
9->36:549.885
10->47:564.004
11->44:651.559
12->41:561.923
13->51:502.759
14->32:455.400
15->40:567.426
16->50:542.542
17->40:456.574
18->43:562.066
19->48:581.938
20->45:528.026
21->33:516.543
22->46:502.264
23->49:590.142
24->42:603.019
25->34:476.562
26->44:474.441
27->36:498.115
28->38:592.224
29->50:531.458
30->33:520.457
31->35:526.102
32->0:578.600
32->14:455.400
32->35:74.898
33->21:516.543
33->30:520.457
33->54:257.232
34->4:630.438
34->25:476.562
34->55:226.768
35->31:526.102
35->32:74.898
35->42:91.481
36->9:549.885
36->27:498.115
36->43:84.434
37->6:523.430
37->8:508.570
37->47:86.496
38->3:483.776
38->28:592.224
38->49:77.358
39->5:559.723
39->7:490.277
39->56:153.630
40->15:567.426
40->17:456.574
40->51:100.241
41->2:468.077
41->12:561.923
41->52:131.850
42->24:603.019
42->35:91.481
42->60:176.021
43->18:562.066
43->36:84.434
43->48:42.812
44->11:6